# CKP02 — DocMind RAG | DecorAI

Pipeline RAG para decoração residencial, ergonomia, iluminação, cores e acabamentos. O notebook compara chunks de 512 e 1024 caracteres, com e sem reranking, e avalia as respostas com RAGAS.

**Observação técnica:** neste notebook, `nomic-embed-text` é executado localmente no ambiente Colab por meio do Ollama; o modelo de geração e o LLM avaliador usam `gemma4:cloud` no Ollama Cloud.

# Dependencias

In [1]:
%pip install -q \
    langchain-core \
    langchain-text-splitters \
    langchain-ollama \
    langchain-chroma \
    "langchain-community==0.4.1" \
    chromadb \
    pymupdf \
    pandas \
    requests \
    tqdm \
    sentence-transformers \
    ragas \
    openai

!apt-get update -qq
!apt-get install -y zstd

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1

In [2]:
import os
import re
import time
import subprocess
import shutil
from pathlib import Path

import requests
import pymupdf
import chromadb
import pandas as pd
import torch
from tqdm.auto import tqdm

from google.colab import userdata
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from sentence_transformers import CrossEncoder
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import Faithfulness, AnswerRelevancy

print("Ambiente preparado!")
print("PyMuPDF:", pymupdf.VersionBind)
print("ChromaDB:", chromadb.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Ambiente preparado!
PyMuPDF: 1.28.2
ChromaDB: 1.5.9
Pandas: 2.2.3
PyTorch: 2.11.0+cu130
CUDA disponível: True
GPU: Tesla T4


# OLLAMA_API_KEY

In [3]:
OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")

if not OLLAMA_API_KEY:
    raise ValueError("Configure OLLAMA_API_KEY nos Secrets do Colab.")

print("Chave carregada com sucesso!")

Chave carregada com sucesso!


# PDF LOADER

In [4]:
PASTA_DOCS = Path("/content/docs")

if not PASTA_DOCS.exists():
    raise FileNotFoundError(
        "A pasta /content/docs não foi encontrada. "
        "Envie ou monte a pasta com os cinco PDFs antes de executar."
    )

arquivos_pdf = sorted(
    arquivo for arquivo in PASTA_DOCS.iterdir()
    if arquivo.is_file() and arquivo.suffix.lower() == ".pdf"
)

print("Total de PDFs encontrados:", len(arquivos_pdf))
for arquivo in arquivos_pdf:
    print("-", arquivo.name)

if len(arquivos_pdf) < 5:
    raise ValueError(f"Esperados pelo menos 5 PDFs; encontrados {len(arquivos_pdf)}.")

documentos = {}
documentos_por_pagina = []

for arquivo in arquivos_pdf:
    with pymupdf.open(arquivo) as pdf:
        textos_paginas = []
        for numero_pagina, pagina in enumerate(pdf, start=1):
            texto = pagina.get_text("text").strip()
            textos_paginas.append(texto)
            if texto:
                documentos_por_pagina.append(
                    Document(
                        page_content=texto,
                        metadata={
                            "source": arquivo.name,
                            "page": numero_pagina,
                            "categoria": "decoracao_residencial",
                            "tipo": "pdf",
                        },
                    )
                )

    texto_completo = "\n".join(textos_paginas)
    documentos[arquivo.name] = {
        "texto": texto_completo,
        "paginas": len(textos_paginas),
        "caracteres": len(texto_completo),
    }
    print(f"{arquivo.name}: {len(textos_paginas)} páginas | {len(texto_completo)} caracteres")

print("Páginas com texto:", len(documentos_por_pagina))

Total de PDFs encontrados: 5
- Acabamentos.pdf
- Estilos_em_Decoracao_ABNT.pdf
- ManualdeiluminacaoeficienteProcel.pdf
- ergonomia.pdf
- psicologia-das-cores.pdf
Acabamentos.pdf: 87 páginas | 34835 caracteres
Estilos_em_Decoracao_ABNT.pdf: 5 páginas | 7831 caracteres
ManualdeiluminacaoeficienteProcel.pdf: 54 páginas | 76330 caracteres
ergonomia.pdf: 215 páginas | 458843 caracteres
psicologia-das-cores.pdf: 39 páginas | 24204 caracteres
Páginas com texto: 396


## 5. Divisão em chunks (`split`) — comparação 512 × 1024

In [5]:
SEPARADORES = ["\n\n", "\n", ". ", " ", ""]

splitter_512 = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
    separators=SEPARADORES,
    strip_whitespace=True,
    add_start_index=True,
)
splitter_1024 = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=128,
    separators=SEPARADORES,
    strip_whitespace=True,
    add_start_index=True,
)

chunks_512 = splitter_512.split_documents(documentos_por_pagina)
chunks_1024 = splitter_1024.split_documents(documentos_por_pagina)

print("Chunks de 512:", len(chunks_512))
print("Chunks de 1024:", len(chunks_1024))

Chunks de 512: 1482
Chunks de 1024: 851


## 6. Inicialização do Ollama local para embeddings

In [6]:
URL_OLLAMA_LOCAL = "http://127.0.0.1:11434"

if not shutil.which("ollama"):
    subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        check=True,
    )

try:
    resposta = requests.get(f"{URL_OLLAMA_LOCAL}/api/tags", timeout=3)
    resposta.raise_for_status()
    print("Ollama já está respondendo.")
except requests.RequestException:
    with open("/content/ollama.log", "w") as log_ollama:
        processo_ollama = subprocess.Popen(
            ["ollama", "serve"],
            stdout=log_ollama,
            stderr=subprocess.STDOUT,
        )
    for tentativa in range(15):
        try:
            resposta = requests.get(f"{URL_OLLAMA_LOCAL}/api/tags", timeout=2)
            if resposta.status_code == 200:
                print("Servidor Ollama iniciado com sucesso!")
                break
        except requests.RequestException:
            pass
        time.sleep(2)
    else:
        raise RuntimeError("O Ollama não iniciou. Confira /content/ollama.log.")

subprocess.run(["ollama", "pull", "nomic-embed-text"], check=True)

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url=URL_OLLAMA_LOCAL,
)

vetor_teste = embeddings.embed_query(
    "A iluminação adequada contribui para o conforto visual em ambientes residenciais."
)
print("Embedding funcionando. Dimensões:", len(vetor_teste))

Servidor Ollama iniciado com sucesso!
Embedding funcionando. Dimensões: 768


In [7]:
import shutil
import subprocess

print("Executável Ollama:", shutil.which("ollama"))

resultado = subprocess.run(
    ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
    capture_output=True,
    text=True
)

print("Código de saída:", resultado.returncode)
print("Saída:", resultado.stdout)
print("Erros:", resultado.stderr)

Executável Ollama: /usr/local/bin/ollama
Código de saída: 0
Saída: WARNING: systemd is not running

Erros: >>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
#=#=#                                                                          
##O#-#                                                                         
##O=#  #                                                                       

                                                                           0.1%
                                                                           0.8%
##                                                                         2.9%
###                                                                        5.1%
#####                                                                      7.2%
######                                                                     8.4%
#######                                 

In [8]:
## 7. Vetorização e armazenamento (`embed` → `store`)

In [9]:
cliente_chroma = chromadb.PersistentClient(path="/content/chroma_db")
TAMANHO_LOTE = 20

def armazenar_chunks(chunks, colecao, tamanho, categoria_desc):
    ids_existentes = set()
    if colecao.count() > 0:
        ids_existentes = set(colecao.get(include=[])['ids'])

    indices_pendentes = [
        i for i in range(len(chunks))
        if f"chunk_{tamanho}_{i}" not in ids_existentes
    ]
    print(f"{categoria_desc}: {len(chunks)} chunks; pendentes: {len(indices_pendentes)}")

    for inicio in tqdm(range(0, len(indices_pendentes), TAMANHO_LOTE), desc=f"Embeddings {tamanho}"):
        indices_lote = indices_pendentes[inicio:inicio + TAMANHO_LOTE]
        lote = [chunks[i] for i in indices_lote]
        textos = [doc.page_content for doc in lote]
        metadados = [
            {
                "source": str(doc.metadata["source"]),
                "page": int(doc.metadata["page"]),
                "categoria": str(doc.metadata.get("categoria", "decoracao_residencial")),
                "tipo": str(doc.metadata.get("tipo", "pdf")),
                "chunk_size": int(tamanho),
                "start_index": int(doc.metadata.get("start_index", 0)),
            }
            for doc in lote
        ]
        ids = [f"chunk_{tamanho}_{i}" for i in indices_lote]
        vetores = embeddings.embed_documents(textos)
        colecao.upsert(ids=ids, documents=textos, embeddings=vetores, metadatas=metadados)

    print(f"Total armazenado na coleção {tamanho}: {colecao.count()}")

colecao_512 = cliente_chroma.get_or_create_collection(
    name="decorai_chunks_512", metadata={"hnsw:space": "cosine"}
)
colecao_1024 = cliente_chroma.get_or_create_collection(
    name="decorai_chunks_1024", metadata={"hnsw:space": "cosine"}
)

armazenar_chunks(chunks_512, colecao_512, 512, "Chunks de 512")
armazenar_chunks(chunks_1024, colecao_1024, 1024, "Chunks de 1024")

Chunks de 512: 1482 chunks; pendentes: 1482


Embeddings 512:   0%|          | 0/75 [00:00<?, ?it/s]

Total armazenado na coleção 512: 1482
Chunks de 1024: 851 chunks; pendentes: 851


Embeddings 1024:   0%|          | 0/43 [00:00<?, ?it/s]

Total armazenado na coleção 1024: 851


## 8. Teste de recuperação vetorial (`retrieve`)

In [10]:
def recuperar_contexto(pergunta, colecao, k=5):
    vetor_pergunta = embeddings.embed_query(pergunta)
    resultados = colecao.query(
        query_embeddings=[vetor_pergunta],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    contextos = []
    for i, texto in enumerate(resultados["documents"][0]):
        metadados = resultados["metadatas"][0][i]
        contextos.append({
            "texto": texto,
            "source": metadados["source"],
            "page": metadados["page"],
            "start_index": metadados.get("start_index", 0),
            "chunk_size": metadados.get("chunk_size", "?"),
            "distance": resultados["distances"][0][i],
        })
    return contextos

pergunta_teste = "Como a iluminação influencia o conforto em ambientes residenciais?"
contextos_512 = recuperar_contexto(pergunta_teste, colecao_512, k=5)
contextos_1024 = recuperar_contexto(pergunta_teste, colecao_1024, k=5)

for nome, contextos in [("512", contextos_512), ("1024", contextos_1024)]:
    print(f"\n=== RECUPERAÇÃO — CHUNKS {nome} ===")
    for i, item in enumerate(contextos, start=1):
        print(f"{i}. {item['source']} | p. {item['page']} | distância: {item['distance']:.4f}")
        print(item["texto"][:300].replace("\n", " "))


=== RECUPERAÇÃO — CHUNKS 512 ===
1. ergonomia.pdf | p. 172 | distância: 0.2132
permitir  algumas  atividades isoladas. Este é um ambiente a  ser usados por grupos. A flexibilidade da  iluminação permitirá o desenvolvimento de  várias atividades simultaneamente           • O comando da iluminação deve ser  previsto de maneira que o usuário entre no  ambiente com as luzes acesas
2. ManualdeiluminacaoeficienteProcel.pdf | p. 32 | distância: 0.2181
Ela define, em muitos casos, as características de um ambiente. Em resumo, ao se projetar a  iluminação de um ambiente, não se deve levar em conta unicamente os aspectos quantitativos,  mas também os qualitativos, de modo a criar uma iluminação que responda a todos os requisitos  que o usuário exige
3. ergonomia.pdf | p. 61 | distância: 0.2196
incidência solar e contato com o ambiente exterior.    Os fatores de conforto exigidos estão relacionados pela atividade ou  grupos de atividades    − Iluminação natural e artificial  − Obscurecimento  − 

## 9. Modelo de geração e prompt com citações de chunks

In [11]:
llm = ChatOllama(
    model="gemma4:cloud",
    temperature=0,
    base_url="https://ollama.com",
    client_kwargs={"headers": {"Authorization": f"Bearer {OLLAMA_API_KEY}"}},
)

prompt_rag = """
<persona>
Você é o DecorAI, um assistente especializado em decoração residencial,
ergonomia, iluminação, cores e acabamentos.
</persona>
<instrucoes>
1. Responda em português brasileiro, com clareza e objetividade.
2. Use exclusivamente as informações presentes no contexto fornecido.
3. Sempre que apresentar uma informação factual baseada no contexto, cite a origem no formato:
[arquivo.pdf, p. X, chunk_id: IDENTIFICADOR_EXATO]
4. Copie o identificador exatamente como aparece no contexto.
5. Não invente dados, referências, páginas ou identificadores de chunks.
6. Se o contexto não for suficiente, informe explicitamente essa limitação.
7. Diferencie informações documentadas de inferências.
8. Não afirme que consultou o documento inteiro quando recebeu apenas trechos.
9. Coloque as citações junto às afirmações que elas sustentam.
</instrucoes>
<contexto>
{contexto}
</contexto>
<pergunta>
{pergunta}
</pergunta>
"""
prompt = ChatPromptTemplate.from_template(prompt_rag)

def identificar_chunk(doc):
    fonte = doc.metadata.get("source", "fonte_desconhecida")
    pagina = doc.metadata.get("page", "?")
    inicio = doc.metadata.get("start_index", "?")
    tamanho = doc.metadata.get("chunk_size", "?")
    return f"{fonte}::p{pagina}::i{inicio}::cs{tamanho}"

def formatar_contexto(docs) -> str:
    partes = []
    for doc in docs:
        fonte = doc.metadata.get("source", "fonte_desconhecida")
        pagina = doc.metadata.get("page", "?")
        partes.append(
            f"[CHUNK_ID: {identificar_chunk(doc)}]\n"
            f"[FONTE: {fonte}, página {pagina}]\n"
            f"{doc.page_content}"
        )
    return "\n\n---\n\n".join(partes)

print(llm.invoke("Responda apenas: conexão com o DecorAI funcionando.").content)

conexão com o DecorAI funcionando.


## 10. Chains RAG com LCEL

In [12]:
db_512 = Chroma(
    collection_name="decorai_chunks_512",
    embedding_function=embeddings,
    persist_directory="/content/chroma_db",
)
db_1024 = Chroma(
    collection_name="decorai_chunks_1024",
    embedding_function=embeddings,
    persist_directory="/content/chroma_db",
)

retriever_512 = db_512.as_retriever(search_type="similarity", search_kwargs={"k": 5})
retriever_1024 = db_1024.as_retriever(search_type="similarity", search_kwargs={"k": 5})

def criar_chain_rag(retriever):
    return (
        {
            "contexto": retriever | RunnableLambda(formatar_contexto),
            "pergunta": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

chain_512 = criar_chain_rag(retriever_512)
chain_1024 = criar_chain_rag(retriever_1024)

print("Chunks armazenados:", db_512._collection.count(), "(512) |", db_1024._collection.count(), "(1024)")

Chunks armazenados: 1482 (512) | 851 (1024)


## 11. Teste comparativo das respostas sem reranking

In [13]:
pergunta_teste = "Como a iluminação influencia o conforto em ambientes residenciais?"

print("=" * 70)
print("DECORAI — CHUNKS 512, SEM RERANKING")
print("=" * 70)
print(chain_512.invoke(pergunta_teste))

print("\n" + "=" * 70)
print("DECORAI — CHUNKS 1024, SEM RERANKING")
print("=" * 70)
print(chain_1024.invoke(pergunta_teste))

DECORAI — CHUNKS 512, SEM RERANKING
Com base no contexto fornecido, a iluminação influencia o conforto e a funcionalidade dos ambientes residenciais da seguinte forma:

*   **Fatores de Conforto:** A iluminação, tanto natural quanto artificial, é listada como um dos fatores de conforto exigidos, os quais estão relacionados à atividade ou grupos de atividades realizados [ergonomia.pdf, p. 61, chunk_id: ergonomia.pdf::p61::i416::cs512].
*   **Qualidade e Requisitos:** A iluminação define as características de um ambiente. Ao projetá-la, não devem ser considerados apenas aspectos quantitativos, mas também qualitativos, para que a iluminação responda a todos os requisitos exigidos pelo usuário do espaço [ManualdeiluminacaoeficienteProcel.pdf, p. 32, chunk_id: ManualdeiluminacaoeficienteProcel.pdf::p32::i442::cs512].
*   **Funcionalidade e Flexibilidade:** A flexibilidade da iluminação permite que várias atividades sejam desenvolvidas simultaneamente em um mesmo ambiente [ergonomia.pdf, p. 

## 12. Reranking com CrossEncoder

In [14]:
MODELO_RERANKER = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(
    MODELO_RERANKER,
    device="cuda" if torch.cuda.is_available() else "cpu",
    max_length=512,
)
print("Reranker carregado:", MODELO_RERANKER)
print("Dispositivo:", reranker.model.device)

retriever_512_candidatos = db_512.as_retriever(
    search_type="similarity", search_kwargs={"k": 10}
)
retriever_1024_candidatos = db_1024.as_retriever(
    search_type="similarity", search_kwargs={"k": 10}
)

def recuperar_e_ranquear(pergunta, retriever, top_n=5):
    candidatos = retriever.invoke(pergunta)
    if not candidatos:
        return []
    pares = [(pergunta, doc.page_content) for doc in candidatos]
    scores = reranker.predict(pares, batch_size=8, show_progress_bar=False)
    ranqueados = sorted(
        zip(candidatos, scores),
        key=lambda item: float(item[1]),
        reverse=True,
    )
    return [doc for doc, score in ranqueados[:top_n]]

def criar_chain_com_reranking(retriever):
    return (
        {
            "contexto": (
                RunnablePassthrough()
                | RunnableLambda(lambda pergunta: recuperar_e_ranquear(pergunta, retriever, top_n=5))
                | RunnableLambda(formatar_contexto)
            ),
            "pergunta": RunnablePassthrough(),
        }
        | prompt
        | llm
        | StrOutputParser()
    )

chain_rerank_512 = criar_chain_com_reranking(retriever_512_candidatos)
chain_rerank_1024 = criar_chain_com_reranking(retriever_1024_candidatos)
print("Chains com reranking criadas.")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker carregado: BAAI/bge-reranker-v2-m3
Dispositivo: cuda:0
Chains com reranking criadas.


## 13. Teste e comparação do reranking

In [15]:
def comparar_rankings(pergunta, retriever, nome_configuracao, top_n=5):
    candidatos = retriever.invoke(pergunta)
    pares = [(pergunta, doc.page_content) for doc in candidatos]
    scores = reranker.predict(pares, batch_size=8, show_progress_bar=False)
    ranqueados = sorted(zip(candidatos, scores), key=lambda item: float(item[1]), reverse=True)

    print(f"\n{'=' * 70}\n{nome_configuracao} — ORDEM ORIGINAL\n{'=' * 70}")
    for i, doc in enumerate(candidatos[:top_n], start=1):
        print(f"{i}. {doc.metadata.get('source', '?')} | p. {doc.metadata.get('page', '?')} | chunk: {identificar_chunk(doc)}")
        print(doc.page_content[:220].replace("\n", " "))

    print(f"\n{'=' * 70}\n{nome_configuracao} — APÓS RERANKING\n{'=' * 70}")
    for i, (doc, score) in enumerate(ranqueados[:top_n], start=1):
        print(f"{i}. score={float(score):.4f} | {doc.metadata.get('source', '?')} | p. {doc.metadata.get('page', '?')} | chunk: {identificar_chunk(doc)}")
        print(doc.page_content[:220].replace("\n", " "))

pergunta_teste = "Como a iluminação influencia o conforto em ambientes residenciais?"
comparar_rankings(pergunta_teste, retriever_512_candidatos, "CHUNKS 512")
comparar_rankings(pergunta_teste, retriever_1024_candidatos, "CHUNKS 1024")

print("\n" + "=" * 70 + "\nRESPOSTA — 512 COM RERANKING\n" + "=" * 70)
print(chain_rerank_512.invoke(pergunta_teste))
print("\n" + "=" * 70 + "\nRESPOSTA — 1024 COM RERANKING\n" + "=" * 70)
print(chain_rerank_1024.invoke(pergunta_teste))


CHUNKS 512 — ORDEM ORIGINAL
1. ergonomia.pdf | p. 172 | chunk: ergonomia.pdf::p172::i412::cs512
permitir  algumas  atividades isoladas. Este é um ambiente a  ser usados por grupos. A flexibilidade da  iluminação permitirá o desenvolvimento de  várias atividades simultaneamente           • O comando da iluminação de
2. ManualdeiluminacaoeficienteProcel.pdf | p. 32 | chunk: ManualdeiluminacaoeficienteProcel.pdf::p32::i442::cs512
Ela define, em muitos casos, as características de um ambiente. Em resumo, ao se projetar a  iluminação de um ambiente, não se deve levar em conta unicamente os aspectos quantitativos,  mas também os qualitativos, de mod
3. ergonomia.pdf | p. 61 | chunk: ergonomia.pdf::p61::i416::cs512
incidência solar e contato com o ambiente exterior.    Os fatores de conforto exigidos estão relacionados pela atividade ou  grupos de atividades    − Iluminação natural e artificial  − Obscurecimento  − Ambiente térmico
4. ergonomia.pdf | p. 85 | chunk: ergonomia.pdf::p85::i1275:

## 14. Validação das citações de chunk

In [16]:
def validar_citacoes(pergunta, chain, retriever, nome_configuracao):
    resposta = chain.invoke(pergunta)
    docs = retriever.invoke(pergunta)
    ids_disponiveis = {identificar_chunk(doc) for doc in docs}

    print(f"\n{'=' * 70}\n{nome_configuracao}\n{'=' * 70}")
    print("RESPOSTA:\n", resposta)
    print("\nCHUNKS RECUPERADOS:")
    for chunk_id in sorted(ids_disponiveis):
        print("-", chunk_id)

    ids_citados = re.findall(r"chunk_id:\s*([^\]\n]+)", resposta)
    if not ids_citados:
        print("\nAtenção: nenhuma citação com chunk_id foi encontrada no formato esperado.")
    else:
        print("\nValidação das citações:")
        for chunk_id in ids_citados:
            chunk_id = chunk_id.strip()
            print(("OK" if chunk_id in ids_disponiveis else "NÃO ENCONTRADO"), chunk_id)
    return resposta

pergunta_citacao = "Como a iluminação influencia o conforto em ambientes residenciais?"
validar_citacoes(pergunta_citacao, chain_512, retriever_512, "512 SEM RERANKING")
validar_citacoes(pergunta_citacao, chain_1024, retriever_1024, "1024 SEM RERANKING")
validar_citacoes(pergunta_citacao, chain_rerank_512, retriever_512_candidatos, "512 COM RERANKING")
validar_citacoes(pergunta_citacao, chain_rerank_1024, retriever_1024_candidatos, "1024 COM RERANKING")


512 SEM RERANKING
RESPOSTA:
 Com base nos documentos fornecidos, a iluminação influencia o conforto residencial da seguinte forma:

*   **Qualidade e Requisitos:** A iluminação define, em muitos casos, as características de um ambiente. Seu projeto não deve considerar apenas aspectos quantitativos, mas também qualitativos, para atender a todos os requisitos exigidos pelo usuário do espaço [ManualdeiluminacaoeficienteProcel.pdf, p. 32, chunk_id: ManualdeiluminacaoeficienteProcel.pdf::p32::i442::cs512].
*   **Fatores de Conforto:** A iluminação (tanto natural quanto artificial) é listada como um dos fatores de conforto exigidos, os quais estão relacionados à atividade ou grupos de atividades realizados [ergonomia.pdf, p. 61, chunk_id: ergonomia.pdf::p61::i416::cs512].
*   **Funcionalidade e Segurança:**
    *   A flexibilidade da iluminação permite que diversas atividades sejam desenvolvidas simultaneamente em um mesmo ambiente [ergonomia.pdf, p. 172, chunk_id: ergonomia.pdf::p172::i412

'Com base nos documentos fornecidos, a influência da iluminação no conforto em ambientes residenciais é abordada da seguinte forma:\n\n*   **Fatores de Conforto:** As exigências de conforto em habitações referem-se às condições de visibilidade, incidência solar e contato com o ambiente exterior, estando a iluminação (tanto natural quanto artificial) e o obscurecimento listados como fatores de conforto exigidos e relacionados às atividades do usuário [ergonomia.pdf, p. 61, chunk_id: ergonomia.pdf::p61::i0::cs1024].\n*   **Qualidade da Iluminação:** Para usuários enfermos, a qualidade da iluminação depende da limitação do brilho incômodo e do ofuscamento (que é considerado prejudicial), além da adequação ao tipo de atividade prevista [ergonomia.pdf, p. 37, chunk_id: ergonomia.pdf::p37::i946::cs1024].\n*   **Aspectos Qualitativos e Globais:** A iluminação deve ser parte de um projeto global e harmonizado, não considerando apenas aspectos quantitativos, mas também qualitativos, para respon

## 15. Perguntas de avaliação


In [17]:
perguntas_avaliacao = [
    {
        "pergunta": "Como a iluminação influencia o conforto em ambientes residenciais?",
        "resposta_referencia": "A iluminação influencia o conforto visual, a percepção das cores e a funcionalidade do ambiente. O planejamento deve considerar a iluminância adequada, a distribuição da luz e a prevenção do ofuscamento.",
    },
    {
        "pergunta": "Como a escolha das cores pode influenciar a percepção de um ambiente?",
        "resposta_referencia": "As cores influenciam a percepção visual do ambiente e sua interação com a iluminação. A aparência das cores pode variar conforme a fonte luminosa.",
    },
    {
        "pergunta": "Quais aspectos ergonômicos devem ser considerados em uma residência?",
        "resposta_referencia": "Devem ser considerados as atividades dos usuários, a funcionalidade, a acessibilidade e as condições de uso dos espaços residenciais.",
    },
    {
        "pergunta": "Como escolher uma iluminação adequada para diferentes atividades?",
        "resposta_referencia": "A iluminação deve considerar a utilização do ambiente, o nível de iluminância necessário, a distribuição da luz e a prevenção do ofuscamento.",
    },
    {
        "pergunta": "Quais são os valores exatos de iluminância, em lux, recomendados para cada cômodo residencial?",
        "resposta_referencia": "Os trechos recuperados da base não fornecem os valores exatos por cômodo. O sistema deve informar essa limitação em vez de inventar valores.",
    },
]

print("Perguntas cadastradas:", len(perguntas_avaliacao))
for i, item in enumerate(perguntas_avaliacao, start=1):
    print(f"{i}. {item['pergunta']}")

Perguntas cadastradas: 5
1. Como a iluminação influencia o conforto em ambientes residenciais?
2. Como a escolha das cores pode influenciar a percepção de um ambiente?
3. Quais aspectos ergonômicos devem ser considerados em uma residência?
4. Como escolher uma iluminação adequada para diferentes atividades?
5. Quais são os valores exatos de iluminância, em lux, recomendados para cada cômodo residencial?


## 16. Gerar respostas para as quatro configurações

In [18]:
def recuperar_textos_avaliacao(pergunta, retriever, usar_reranking=False):
    if usar_reranking:
        docs = recuperar_e_ranquear(pergunta, retriever, top_n=5)
    else:
        docs = retriever.invoke(pergunta)
    return docs, [doc.page_content for doc in docs]

configuracoes_avaliacao = {
    "512_sem_reranking": {"retriever": retriever_512, "reranking": False},
    "1024_sem_reranking": {"retriever": retriever_1024, "reranking": False},
    "512_com_reranking": {"retriever": retriever_512_candidatos, "reranking": True},
    "1024_com_reranking": {"retriever": retriever_1024_candidatos, "reranking": True},
}

registros_avaliacao = []
for nome_config, config in configuracoes_avaliacao.items():
    retriever = config["retriever"]
    usar_reranking = config["reranking"]
    chain = (
        chain_rerank_512 if nome_config == "512_com_reranking" else
        chain_rerank_1024 if nome_config == "1024_com_reranking" else
        chain_512 if nome_config == "512_sem_reranking" else
        chain_1024
    )

    print(f"\nExecutando configuração: {nome_config}")
    for item in tqdm(perguntas_avaliacao, desc=nome_config):
        pergunta = item["pergunta"]
        docs, contextos = recuperar_textos_avaliacao(pergunta, retriever, usar_reranking)
        resposta = chain.invoke(pergunta)
        registros_avaliacao.append({
            "configuracao": nome_config,
            "pergunta": pergunta,
            "resposta_referencia": item["resposta_referencia"],
            "resposta": resposta,
            "contextos": contextos,
            "fontes": [
                {
                    "source": doc.metadata.get("source", "?"),
                    "page": doc.metadata.get("page", "?"),
                    "chunk_id": identificar_chunk(doc),
                }
                for doc in docs
            ],
        })

df_avaliacao = pd.DataFrame(registros_avaliacao)
print("Total de registros:", len(df_avaliacao))
display(df_avaliacao[["configuracao", "pergunta", "resposta"]].head())
print("\nRegistros por configuração:")
display(df_avaliacao.groupby("configuracao").size().to_frame("quantidade"))


Executando configuração: 512_sem_reranking


512_sem_reranking:   0%|          | 0/5 [00:00<?, ?it/s]


Executando configuração: 1024_sem_reranking


1024_sem_reranking:   0%|          | 0/5 [00:00<?, ?it/s]


Executando configuração: 512_com_reranking


...

1024_com_reranking:   0%|          | 0/5 [00:00<?, ?it/s]

Total de registros: 20


,configuracao,pergunta,resposta
0,512_sem_reranking,Como a iluminação influencia o conforto em amb...,"Com base nos documentos fornecidos, a iluminaç..."
1,512_sem_reranking,Como a escolha das cores pode influenciar a pe...,"Com base no contexto fornecido, a escolha das ..."
2,512_sem_reranking,Quais aspectos ergonômicos devem ser considera...,"Com base no contexto fornecido, os aspectos er..."
3,512_sem_reranking,Como escolher uma iluminação adequada para dif...,Para escolher uma iluminação adequada para dif...
4,512_sem_reranking,"Quais são os valores exatos de iluminância, em...",O contexto fornecido não apresenta os valores ...



Registros por configuração:


,quantidade
configuracao,
1024_com_reranking,5
1024_sem_reranking,5
512_com_reranking,5
512_sem_reranking,5


## 17. Configurar RAGAS

In [19]:
cliente_avaliador = AsyncOpenAI(
    api_key=OLLAMA_API_KEY,
    base_url="https://ollama.com/v1",
)
avaliador_llm = llm_factory(
    "gemma4:cloud",
    provider="openai",
    client=cliente_avaliador,
    max_tokens=4098,
)

# O RAGAS usa o endpoint local compatível com OpenAI para embeddings de avaliação.
cliente_embeddings_avaliacao = AsyncOpenAI(
    api_key="ollama",
    base_url=f"{URL_OLLAMA_LOCAL}/v1",
)
avaliador_embeddings = embedding_factory(
    provider="openai",
    model="nomic-embed-text",
    client=cliente_embeddings_avaliacao,
    interface="modern",
)

metrica_faithfulness = Faithfulness(llm=avaliador_llm)
metrica_answer_relevancy = AnswerRelevancy(
    llm=avaliador_llm,
    embeddings=avaliador_embeddings,
)
print("Métricas RAGAS configuradas.")

Métricas RAGAS configuradas.


## 18. Teste rápido das métricas (opcional)

In [20]:
registro = df_avaliacao.iloc[0]

resultado_faithfulness = await metrica_faithfulness.ascore(
    user_input=registro["pergunta"],
    response=registro["resposta"],
    retrieved_contexts=registro["contextos"],
)
resultado_relevancia = await metrica_answer_relevancy.ascore(
    user_input=registro["pergunta"],
    response=registro["resposta"],
)

print("Pergunta:", registro["pergunta"])
print("Faithfulness:", resultado_faithfulness.value)
print("Answer relevancy:", resultado_relevancia.value)

Pergunta: Como a iluminação influencia o conforto em ambientes residenciais?
Faithfulness: 1.0
Answer relevancy: 0.8840643594288594


## 19. Avaliação RAGAS das 20 respostas

In [21]:
resultados_metricas = []

async def avaliar_respostas():
    concluidos = {
        (item["configuracao"], item["pergunta"])
        for item in resultados_metricas
    }

    for _, registro in tqdm(
        df_avaliacao.iterrows(),
        total=len(df_avaliacao),
        desc="Avaliando respostas",
    ):
        chave = (registro["configuracao"], registro["pergunta"])
        if chave in concluidos:
            continue
        try:
            faithfulness = await metrica_faithfulness.ascore(
                user_input=registro["pergunta"],
                response=registro["resposta"],
                retrieved_contexts=registro["contextos"],
            )
            relevancia = await metrica_answer_relevancy.ascore(
                user_input=registro["pergunta"],
                response=registro["resposta"],
            )
            resultados_metricas.append({
                "configuracao": registro["configuracao"],
                "pergunta": registro["pergunta"],
                "faithfulness": float(faithfulness.value),
                "answer_relevancy": float(relevancia.value),
            })
            concluidos.add(chave)
        except Exception as erro:
            print(f"\nErro em {chave}: {type(erro).__name__}: {erro}")
            break

await avaliar_respostas()
df_metricas = pd.DataFrame(resultados_metricas)
print(f"Avaliações concluídas: {len(df_metricas)}/{len(df_avaliacao)}")
display(df_metricas)

Avaliando respostas:   0%|          | 0/20 [00:00<?, ?it/s]

Avaliações concluídas: 20/20


,configuracao,pergunta,faithfulness,answer_relevancy
0,512_sem_reranking,Como a iluminação influencia o conforto em amb...,1.000000,0.884064
1,512_sem_reranking,Como a escolha das cores pode influenciar a pe...,1.000000,0.912013
2,512_sem_reranking,Quais aspectos ergonômicos devem ser considera...,1.000000,0.940637
3,512_sem_reranking,Como escolher uma iluminação adequada para dif...,1.000000,0.882351
4,512_sem_reranking,"Quais são os valores exatos de iluminância, em...",1.000000,0.000000
5,1024_sem_reranking,Como a iluminação influencia o conforto em amb...,0.916667,0.912643
6,1024_sem_reranking,Como a escolha das cores pode influenciar a pe...,1.000000,0.992755
7,1024_sem_reranking,Quais aspectos ergonômicos devem ser considera...,1.000000,0.919074
8,1024_sem_reranking,Como escolher uma iluminação adequada para dif...,1.000000,0.882351
9,1024_sem_reranking,"Quais são os valores exatos de iluminância, em...",1.000000,0.000000


## 20. Resumo das métricas


In [22]:
resumo_metricas = (
    df_metricas.groupby("configuracao")
    .agg(
        quantidade=("pergunta", "count"),
        faithfulness_media=("faithfulness", "mean"),
        relevancia_media=("answer_relevancy", "mean"),
        faithfulness_min=("faithfulness", "min"),
        relevancia_min=("answer_relevancy", "min"),
        relevancia_mediana=("answer_relevancy", "median"),
        respostas_com_relevancia_zero=("answer_relevancy", lambda notas: (notas == 0).sum()),
    )
    .reset_index()
)
display(resumo_metricas.round(4))

,configuracao,quantidade,faithfulness_media,relevancia_media,faithfulness_min,relevancia_min,relevancia_mediana,respostas_com_relevancia_zero
0,1024_com_reranking,5,1.0000,0.7719,1.0000,0.0,0.9662,1
1,1024_sem_reranking,5,0.9833,0.7414,0.9167,0.0,0.9126,1
2,512_com_reranking,5,1.0000,0.7564,1.0000,0.0,0.9247,1
3,512_sem_reranking,5,1.0000,0.7238,1.0000,0.0,0.8841,1


## 21. Exportar resultados


In [23]:
resumo_metricas.to_csv(
    "/content/resumo_metricas_ragas.csv", index=False, encoding="utf-8-sig"
)
df_metricas.to_csv(
    "/content/avaliacoes_ragas.csv", index=False, encoding="utf-8-sig"
)
df_avaliacao.to_json(
    "/content/respostas_avaliadas.json", orient="records", force_ascii=False, indent=2
)
print("Arquivos salvos em /content:")
print("- resumo_metricas_ragas.csv")
print("- avaliacoes_ragas.csv")
print("- respostas_avaliadas.json")

Arquivos salvos em /content:
- resumo_metricas_ragas.csv
- avaliacoes_ragas.csv
- respostas_avaliadas.json
